# Evaluate Development data

## Set Up

In [ ]:
import os
import sys

sys.path.append(os.path.abspath("../"))
from src.config import BASE_PATH
from src.data_utils import get_data, get_models
from src.eval import evaluate_models, NumpyEncoder, BIN_NAMES
import pandas as pd
import json
import numpy as np
import warnings
from shutil import rmtree

print(f"Path: {BASE_PATH}")

Globals

In [ ]:
# Data
DATA_DICT = {"base": get_data(is_nomo=False), "nomo": get_data(is_nomo=True)}

# Models
model_dir = BASE_PATH / "models" / "trained"
model_prefix_list = ["lgbm", "xgb", "knn", "svc", "nn", "stack"]
base_model_dict = {}

## Base models
base_model_dict = get_models(model_prefix_list, model_dir)
## Nomogram
nomo_model_dict = get_models(["lr"], model_dir)

## Evaluate

- Use validation as placeholder for test

In [ ]:
all_models_test_dict = {}
ALL_DICT = {}
cls_rows = []
cls_index = []
# n_bootstraps = 5000
n_bootstraps = 5
save_path = BASE_PATH / "results"

Base

In [ ]:
## BASE MODELS
# set n bins in src/eval.py
class_report_dict, bin_report_dict = evaluate_models(
    model_dict=base_model_dict,
    X_train=DATA_DICT["base"]["X_train"],
    y_train=DATA_DICT["base"]["y_train"].values.ravel(),
    X_val=DATA_DICT["base"]["X_test"],
    y_val=DATA_DICT["base"]["y_test"].values.ravel(),
    X_test=DATA_DICT["base"]["X_test"],  # use placeholds for now
    y_test=DATA_DICT["base"]["y_test"].values.ravel(),  # use placeholds for now
    results_path=save_path,
    threshold_str="val",
    show_cm=False,
    show_roc=False,
    show_cal=False,
    n_bootstraps=n_bootstraps,
    show_progress=False,
)
## ONLY export test (have access to train/val if need be)
for model, metrics in class_report_dict["test"].items():
    cls_rows.append(metrics)
    cls_index.append(model)

Nomogram

In [ ]:
## Nomogram (LR) MODEL
nomo_class_report_dict, nomo_bin_report_dict = evaluate_models(
    model_dict=nomo_model_dict,
    X_train=DATA_DICT["nomo"]["X_train"],
    y_train=DATA_DICT["nomo"]["y_train"].values.ravel(),
    X_val=DATA_DICT["nomo"]["X_test"],
    y_val=DATA_DICT["nomo"]["y_test"].values.ravel(),
    X_test=DATA_DICT["nomo"]["X_test"],  # use placeholds for now
    y_test=DATA_DICT["nomo"]["y_test"].values.ravel(),  # use placeholds for now
    results_path=save_path,
    threshold_str="val",
    show_cm=False,
    show_roc=False,
    show_cal=False,
    n_bootstraps=n_bootstraps,
    show_progress=False,
)
# Class
ALL_DICT["class"] = class_report_dict
ALL_DICT["class"]["train"]["lr"] = nomo_class_report_dict["train"]
ALL_DICT["class"]["val"]["lr"] = nomo_class_report_dict["val"]
ALL_DICT["class"]["test"]["lr"] = nomo_class_report_dict["test"]
# Bins
bin_report_dict["lr"] = nomo_bin_report_dict["lr"]
ALL_DICT["bins"] = bin_report_dict
## ONLY export test (have access to train/val if need be)
for model, metrics in nomo_class_report_dict["test"].items():
    cls_rows.append(metrics)
    cls_index.append(model)

Save all results in JSON

In [ ]:
all_models_outcomes_df = pd.DataFrame(cls_rows, index=cls_index)
all_save_path = save_path / "tables" / "all_dict_results.json"
if all_save_path.exists():
    all_save_path.unlink()
all_save_path.parent.mkdir(exist_ok=True, parents=True)
with open(all_save_path, "w") as f:
    json.dump(ALL_DICT, f, cls=NumpyEncoder, indent=2)

Re-format bins

In [ ]:
# Load it back
with open(all_save_path, "r") as f:
    loaded_dict = json.load(f)
bin_rows = []
bin_index = []
bins_dict = loaded_dict["bins"]
for model_name, bins in bins_dict.items():
    for bin_name, metrics in bins.items():

        # Extract n and percentage
        n_perc = metrics["n_perc"]
        n = n_perc["n"]
        perc_cohort = n_perc["perc"]

        # Extract percentage of all positives
        perc_all_pos = metrics["perc_all_pos"]
        n_pos = perc_all_pos["n"]
        perc_pos = perc_all_pos["perc"]

        # Extract event rate with CIs
        event_dict = metrics["event_rate_w_CIs"]
        event_rate = event_dict["event_rate"]
        ci_lower = event_dict.get("lower_CI", "N/A")
        ci_upper = event_dict.get("upper_CI", "N/A")

        # Format event rate string
        ci_str = f"({ci_lower:.2%}, {ci_upper:.2%})"

        # Extract lift
        lift = metrics["lift"]

        # Extract thresholds and mean output
        thresholds = metrics["thresholds"]
        mean_output = metrics["mean_model_output"]

        # Build row
        row = {
            "N (% of tot cohort)": f"{int(n)} ({float(perc_cohort):.2%})",
            "N pos (% of All Positives)": f"{int(n_pos)} ({float(perc_pos):.2%})",
            "Event Rate (95% CI)": f"{event_rate:.2%} {ci_str}",
            "Lift": f"{float(lift):.2f}" if not np.isnan(lift) else np.nan,
            "Thresholds": thresholds,
            "Mean Model Output": (
                f"{float(mean_output):.2%}" if not np.isnan(mean_output) else np.nan
            ),
        }
        bin_rows.append(row)
        bin_index.append((model_name, bin_name))
# Create DataFrame with MultiIndex
df = pd.DataFrame(
    bin_rows, index=pd.MultiIndex.from_tuples(bin_index, names=["Model", "Bin"])
)
# Define custom ordering for Bin level
bin_order = BIN_NAMES

# Convert Bin level to categorical with custom order
df.index = df.index.set_levels(  # type: ignore
    pd.CategoricalIndex(df.index.levels[1], categories=bin_order, ordered=True),  # type: ignore
    level=1,
)

# Sort with the custom ordering
df = df.sort_index()

# Convert to flat table
df_flat = df.reset_index()
df_flat

Export tables

In [ ]:
report_path = save_path / "tables" / "metrics"
if report_path.exists():
    rmtree(report_path)
report_path.mkdir(exist_ok=True, parents=True)
df_flat.to_excel(report_path / "bin_report.xlsx")
all_models_outcomes_df.to_excel(report_path / "class_report.xlsx")